In [0]:
from pyspark.sql import functions as F

BRONZE = "capstone_project_dev.bronze.raw_metadata"
SILVER = "capstone_project_dev.silver.validated_metadata"

print("Setup done")

In [0]:
df = spark.table(BRONZE)

print(f"Rows loaded : {df.count():,}")
print(f"Cols loaded : {len(df.columns)}")
df.show(3, truncate=30)

In [0]:
from pyspark.sql.window import Window

# Add timestamp for tiebreaking
df = df.withColumn("_processed_at", F.current_timestamp())

# Count nulls per row — fewer nulls = more complete row
required_fields = [
    "column_desc", "term_name", "data_steward",
    "security_classification", "certification_level"
]

null_score = sum(
    F.when(F.col(c).isNull(), 1).otherwise(0)
    for c in required_fields
)

df = df.withColumn("_null_score", null_score)

window = Window.partitionBy("table_name", "column_name") \
               .orderBy(F.col("_null_score").asc(), F.col("column_id").asc())

df = df.withColumn("_row_rank", F.row_number().over(window)) \
       .filter(F.col("_row_rank") == 1) \
       .drop("_row_rank", "_null_score")

print(f"Rows after dedup : {df.count():,}")